In [4]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor

# ============================================================
# 1. LOAD DATA
# ============================================================
HepC = pd.read_excel("HepC.xlsx")
HepC.columns = HepC.columns.str.strip()

print("Columns:")
for col in HepC.columns:
    print(repr(col))

# ============================================================
# 2. DEFINE RESPONSE + PREDICTORS
# ============================================================
response = "vunerability population"

predictors = [
    "Female Ages 18-85",
    "Male Ages 18-85",
    "MSM Population Black",
    "MSM population Hispanic",
    "Substance Abuse Rates",
    "Homeless Population",
    "uninsured",
    "100_poverty_level",
    "Incarcerated Populations"
]

# ============================================================
# 3. HELPER FUNCTIONS
# ============================================================
def qterm(name):
    return f'Q("{name}")'

def build_formula(response_var, predictor_list):
    x_part = " + ".join([qterm(p) if not p.isidentifier() else p for p in predictor_list])
    y_part = qterm(response_var) if not response_var.isidentifier() else response_var
    return f"{y_part} ~ {x_part}"

def calculate_vif(df, features):
    X = df[features].copy()
    X = sm.add_constant(X)

    vif_results = []
    for i in range(X.shape[1]):
        try:
            vif_val = variance_inflation_factor(X.values, i)
        except Exception:
            vif_val = np.nan
        vif_results.append(vif_val)

    return pd.DataFrame({
        "Variable": X.columns,
        "VIF": vif_results
    })

full_formula = build_formula(response, predictors)

# ============================================================
# 4. POISSON MODEL
# ============================================================
print("\n" + "=" * 60)
print("POISSON MODEL")
print("=" * 60)

Poisson1 = smf.glm(
    formula=full_formula,
    data=HepC,
    family=sm.families.Poisson()
).fit()

print(Poisson1.summary())

# ============================================================
# 5. VIF
# ============================================================
print("\n" + "=" * 60)
print("VIF")
print("=" * 60)

vif_table = calculate_vif(HepC, predictors)
print(vif_table)

# ============================================================
# 6. NEGATIVE BINOMIAL GLM
#    This is closer to R's glm.nb than smf.negativebinomial()
# ============================================================
print("\n" + "=" * 60)
print("NEGATIVE BINOMIAL GLM")
print("=" * 60)

NegBin1 = smf.glm(
    formula=full_formula,
    data=HepC,
    family=sm.families.NegativeBinomial(alpha=1.0)
).fit()

print(NegBin1.summary())

# ============================================================
# 7. BACKWARD STEPWISE USING GLM NEGATIVE BINOMIAL + AIC
# ============================================================
def backward_stepwise_nb_glm(data, response_var, predictor_list):
    remaining = predictor_list.copy()
    current_formula = build_formula(response_var, remaining)

    current_model = smf.glm(
        formula=current_formula,
        data=data,
        family=sm.families.NegativeBinomial(alpha=1.0)
    ).fit()

    current_aic = current_model.aic

    print("\n" + "=" * 60)
    print("STEPWISE BACKWARD PROCESS (NB GLM)")
    print("=" * 60)
    print(f"Start AIC: {current_aic:.6f}")
    print(f"Start formula: {current_formula}\n")

    step_num = 1

    while len(remaining) > 1:
        candidate_results = []

        for predictor in remaining:
            trial_predictors = [p for p in remaining if p != predictor]
            trial_formula = build_formula(response_var, trial_predictors)

            try:
                trial_model = smf.glm(
                    formula=trial_formula,
                    data=data,
                    family=sm.families.NegativeBinomial(alpha=1.0)
                ).fit()

                if np.isfinite(trial_model.aic):
                    candidate_results.append({
                        "removed": predictor,
                        "aic": trial_model.aic,
                        "formula": trial_formula,
                        "model": trial_model,
                        "predictors": trial_predictors
                    })
            except Exception as e:
                print(f"Could not fit model without '{predictor}': {e}")

        if not candidate_results:
            print("No valid candidate models could be fit. Stopping.")
            break

        candidate_table = pd.DataFrame(
            [{"Removed": r["removed"], "AIC": r["aic"]} for r in candidate_results]
        ).sort_values("AIC")

        print(f"Step {step_num} candidate removals:")
        print(candidate_table.to_string(index=False))
        print()

        best_result = min(candidate_results, key=lambda x: x["aic"])

        if best_result["aic"] < current_aic:
            print(f"Step {step_num}: removing '{best_result['removed']}'")
            print(f"Old AIC: {current_aic:.6f}")
            print(f"New AIC: {best_result['aic']:.6f}\n")

            current_aic = best_result["aic"]
            current_model = best_result["model"]
            current_formula = best_result["formula"]
            remaining = best_result["predictors"]
            step_num += 1
        else:
            print("No further AIC improvement. Stopping.\n")
            break

    return current_model, current_formula, remaining

stepwise_backward2, final_formula, final_predictors = backward_stepwise_nb_glm(
    data=HepC,
    response_var=response,
    predictor_list=predictors
)

print("\n" + "=" * 60)
print("FINAL STEPWISE BACKWARD NB GLM MODEL")
print("=" * 60)
print("Final predictors:")
print(final_predictors)
print("\nFinal formula:")
print(final_formula)
print("\nFinal model summary:")
print(stepwise_backward2.summary())

# ============================================================
# 8. POISSON DISPERSION CHECK
# ============================================================
print("\n" + "=" * 60)
print("POISSON DISPERSION CHECK")
print("=" * 60)

pearson_chi2 = np.sum(Poisson1.resid_pearson ** 2)
dispersion_ratio = pearson_chi2 / Poisson1.df_resid

print(f"Pearson Chi-Square: {pearson_chi2}")
print(f"Residual DF: {Poisson1.df_resid}")
print(f"Dispersion Ratio: {dispersion_ratio}")

if dispersion_ratio > 1.5:
    print("Possible overdispersion detected.")
elif dispersion_ratio < 0.75:
    print("Possible underdispersion detected.")
else:
    print("No strong evidence of overdispersion or underdispersion.")

# ============================================================
# 9. AIC COMPARISON
# ============================================================
print("\n" + "=" * 60)
print("AIC COMPARISON")
print("=" * 60)

print("AIC(Poisson1):", Poisson1.aic)
print("AIC(NegBin1):", NegBin1.aic)
print("AIC(stepwise_backward2):", stepwise_backward2.aic)

# ============================================================
# 10. EXPONENTIATED COEFFICIENTS
# ============================================================
print("\n" + "=" * 60)
print("EXPONENTIATED COEFFICIENTS OF FINAL STEPWISE MODEL")
print("=" * 60)

print(np.exp(stepwise_backward2.params))

Columns:
'vunerability population'
'Female Ages 18-85'
'Male Ages 18-85'
'MSM Population Black'
'MSM population Hispanic'
'Substance Abuse Rates'
'Homeless Population'
'uninsured'
'100_poverty_level'
'Incarcerated Populations'

POISSON MODEL
                      Generalized Linear Model Regression Results                       
Dep. Variable:     Q("vunerability population")   No. Observations:                   54
Model:                                      GLM   Df Residuals:                       45
Model Family:                           Poisson   Df Model:                            8
Link Function:                              Log   Scale:                          1.0000
Method:                                    IRLS   Log-Likelihood:                -128.02
Date:                          Thu, 12 Mar 2026   Deviance:                       23.118
Time:                                  21:12:04   Pearson chi2:                     18.2
No. Iterations:                              3